In [1]:
pip install ucimlrepo

  Obtaining dependency information for ucimlrepo from https://files.pythonhosted.org/packages/3b/07/1252560194df2b4fad1cb3c46081b948331c63eb1bb0b97620d508d12a53/ucimlrepo-0.0.7-py3-none-any.whl.metadata
Note: you may need to restart the kernel to use updated packages.


In [2]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
statlog_german_credit_data = fetch_ucirepo(id=144) 
  
# data (as pandas dataframes) 
X = statlog_german_credit_data.data.features 
y = statlog_german_credit_data.data.targets 
  
# metadata 
print(statlog_german_credit_data.metadata) 
  
# variable information 
print(statlog_german_credit_data.variables) 

C:\Users\Anri Wang\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


{'uci_id': 144, 'name': 'Statlog (German Credit Data)', 'repository_url': 'https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data', 'data_url': 'https://archive.ics.uci.edu/static/public/144/data.csv', 'abstract': 'This dataset classifies people described by a set of attributes as good or bad credit risks. Comes in two formats (one all numeric). Also comes with a cost matrix', 'area': 'Social Science', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 1000, 'num_features': 20, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Other', 'Marital Status', 'Age', 'Occupation'], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1994, 'last_updated': 'Thu Aug 10 2023', 'dataset_doi': '10.24432/C5NC77', 'creators': ['Hans Hofmann'], 'intro_paper': None, 'additional_info': {'summary': 'Two datasets are provided.  the original dataset, in the form provided by

In [3]:
import pandas as pd
import numpy as np

# 如果你已经按 ucimlrepo 的例子跑了：
# X = statlog_german_credit_data.data.features
# y = statlog_german_credit_data.data.targets

# 看看维度
print("X shape:", X.shape)
print("y shape:", y.shape)
print("y head:\n", y.head())

# y 可能是 DataFrame，取出那一列
y_series = y.iloc[:, 0]

# 按说明：1=Good, 2=Bad  →  Bad=1, Good=0
y_bad = (y_series == 2).astype(int)

print("Bad rate:", y_bad.mean())
print(y_series.value_counts())

X shape: (1000, 20)
y shape: (1000, 1)
y head:
    class
0      1
1      2
2      1
3      1
4      2
Bad rate: 0.3
class
1    700
2    300
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report, accuracy_score

# 列类型
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

print("num cols:", len(num_cols), "cat cols:", len(cat_cols))

preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ],
    remainder="drop"
)

model = LogisticRegression(max_iter=2000)

clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", model)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y_bad, test_size=0.2, random_state=42, stratify=y_bad
)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nReport:\n", classification_report(y_test, y_pred))

num cols: 7 cat cols: 13
Accuracy: 0.79
ROC-AUC: 0.8098809523809524
Confusion matrix:
 [[125  15]
 [ 27  33]]

Report:
               precision    recall  f1-score   support

           0       0.82      0.89      0.86       140
           1       0.69      0.55      0.61        60

    accuracy                           0.79       200
   macro avg       0.75      0.72      0.73       200
weighted avg       0.78      0.79      0.78       200



In [5]:
def threshold_report(y_true, y_score, threshold):
    y_hat = (y_score >= threshold).astype(int)  # 1=预测Bad
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat).ravel()

    tpr = tp / (tp + fn) if (tp + fn) else 0.0  # recall for bad
    fpr = fp / (fp + tn) if (fp + tn) else 0.0

    approve_rate = (y_hat == 0).mean()          # 预测Good的比例
    bad_capture = tpr                            # 抓到坏人的比例
    false_decline = fpr                          # 把好人拒了的比例（相对好人群体）

    return {
        "threshold": threshold,
        "approve_rate": approve_rate,
        "TPR_bad_capture": bad_capture,
        "FPR_false_decline": false_decline,
        "TP": tp, "FP": fp, "TN": tn, "FN": fn
    }

for t in [0.2, 0.3, 0.4, 0.5]:
    print(threshold_report(y_test.values, y_proba, t))

{'threshold': 0.2, 'approve_rate': 0.445, 'TPR_bad_capture': 0.85, 'FPR_false_decline': 0.42857142857142855, 'TP': 51, 'FP': 60, 'TN': 80, 'FN': 9}
{'threshold': 0.3, 'approve_rate': 0.565, 'TPR_bad_capture': 0.7833333333333333, 'FPR_false_decline': 0.2857142857142857, 'TP': 47, 'FP': 40, 'TN': 100, 'FN': 13}
{'threshold': 0.4, 'approve_rate': 0.675, 'TPR_bad_capture': 0.6666666666666666, 'FPR_false_decline': 0.17857142857142858, 'TP': 40, 'FP': 25, 'TN': 115, 'FN': 20}
{'threshold': 0.5, 'approve_rate': 0.76, 'TPR_bad_capture': 0.55, 'FPR_false_decline': 0.10714285714285714, 'TP': 33, 'FP': 15, 'TN': 125, 'FN': 27}


In [6]:
def expected_cost(y_true, y_score, threshold, cost_fp=1.0, cost_fn=10.0):
    y_hat = (y_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat).ravel()
    return cost_fp * fp + cost_fn * fn

thresholds = np.linspace(0.05, 0.95, 19)
costs = [expected_cost(y_test.values, y_proba, t, cost_fp=1, cost_fn=10) for t in thresholds]

best_t = thresholds[int(np.argmin(costs))]
print("Best threshold (min cost):", best_t)
print("Min cost:", min(costs))
print("At best threshold:", threshold_report(y_test.values, y_proba, best_t))

Best threshold (min cost): 0.05
Min cost: 126
At best threshold: {'threshold': 0.05, 'approve_rate': 0.125, 'TPR_bad_capture': 0.9833333333333333, 'FPR_false_decline': 0.8285714285714286, 'TP': 59, 'FP': 116, 'TN': 24, 'FN': 1}


In [7]:
pd.Series(y_proba).describe()

count    200.000000
mean       0.313554
std        0.250544
min        0.007096
25%        0.100623
50%        0.261370
75%        0.484529
max        0.940123
dtype: float64

In [8]:
pd.Series(y_proba).quantile([0.05, 0.1, 0.2, 0.5, 0.8, 0.9])

0.05    0.026542
0.10    0.044105
0.20    0.079823
0.50    0.261370
0.80    0.542197
0.90    0.692156
dtype: float64

In [9]:
import numpy as np
from sklearn.metrics import confusion_matrix

def threshold_metrics(y_true, y_score, threshold):
    # 1=预测Bad(拒绝), 0=预测Good(通过)
    y_hat = (y_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat).ravel()

    approve_rate = (y_hat == 0).mean()
    tpr = tp / (tp + fn) if (tp + fn) else 0.0  # 抓坏率（Recall for bad）
    fpr = fp / (fp + tn) if (fp + tn) else 0.0  # 错拒率（对好人而言）

    return tn, fp, fn, tp, approve_rate, tpr, fpr

def expected_cost(fp, fn, cost_fp=1.0, cost_fn=10.0):
    return cost_fp * fp + cost_fn * fn

# 你可以改这个通过率目标
target_approve = 0.60

thresholds = np.linspace(0.01, 0.99, 199)
rows = []

for t in thresholds:
    tn, fp, fn, tp, approve_rate, tpr, fpr = threshold_metrics(y_test.values, y_proba, t)
    cost = expected_cost(fp, fn, cost_fp=1, cost_fn=10)
    rows.append((t, approve_rate, tpr, fpr, fp, fn, cost))

# 先筛选“通过率最接近目标”的候选阈值，再在其中选成本最低
rows = sorted(rows, key=lambda x: abs(x[1] - target_approve))
candidates = rows[:20]  # 通过率最接近的20个阈值
best = min(candidates, key=lambda x: x[-1])

print("Best threshold under approve-rate constraint:")
print("threshold =", best[0])
print("approve_rate =", best[1])
print("TPR_bad_capture =", best[2])
print("FPR_false_decline =", best[3])
print("FP =", best[4], "FN =", best[5])
print("expected_cost =", best[6])

Best threshold under approve-rate constraint:
threshold = 0.2871717171717172
approve_rate = 0.56
TPR_bad_capture = 0.8
FPR_false_decline = 0.2857142857142857
FP = 40 FN = 12
expected_cost = 160


In [11]:
# 把测试集拿出来
test_df = X_test.copy()
test_df["y_true"] = y_test.values
test_df["PD"] = y_proba

# 用最终阈值
final_t = best[0]
test_df["approve"] = (test_df["PD"] < final_t).astype(int)  # 1=通过
test_df["pred_bad"] = (test_df["PD"] >= final_t).astype(int)

# 年龄分桶
test_df["age_group"] = pd.cut(
    test_df["Attribute13"],
    bins=[18, 25, 40, 100],
    labels=["<25", "25-40", "40+"]
)

# 分组看策略表现
group_stats = test_df.groupby("age_group").apply(
    lambda g: pd.Series({
        "approve_rate": g["approve"].mean(),
        "bad_rate": g["y_true"].mean(),
        "TPR_bad_capture": (
            ((g["pred_bad"] == 1) & (g["y_true"] == 1)).sum() /
            (g["y_true"] == 1).sum()
            if (g["y_true"] == 1).sum() > 0 else np.nan
        )
    })
)

print(group_stats)

           approve_rate  bad_rate  TPR_bad_capture
age_group                                         
<25            0.454545  0.454545         0.750000
25-40          0.565217  0.293478         0.851852
40+            0.625000  0.203125         0.769231


C:\Users\Anri Wang\AppData\Local\Temp\ipykernel_11236\1839976498.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_stats = test_df.groupby("age_group").apply(
C:\Users\Anri Wang\AppData\Local\Temp\ipykernel_11236\1839976498.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  group_stats = test_df.groupby("age_group").apply(
